# 🚀 Complete Master Analysis: Language Education in a Brave New World
### Comprehensive Pipeline (EDA, Paired T-Tests, AGS Scale Reliability, and LMEM)
**Author:** Dr. Pegah Merrikhi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'


## 1. Load Data & Demographics

In [ ]:
participants_df = pd.read_csv('participants.csv')
print(f"Total Participants: {len(participants_df)}")
print(participants_df.head())

# Demographics breakdown
print("\n--- Gender Distribution ---")
print(participants_df['Gender_Code'].value_counts())

print("\n--- CEFR Proficiency ---")
print(participants_df['CEFR_Code'].value_counts())

print("\n--- AI Experience (Months) Summary ---")
print(participants_df['AI_Experience_Months'].describe())


## 2. Authenticity Gap Scale (AGS) Reliability

In [ ]:
ags_df = pd.read_csv('ags_survey_items.csv')
item_cols = [f'AGS_{i:02d}' for i in range(1, 16)]
items_data = ags_df[item_cols]

def cronbach_alpha(df_items):
    k = df_items.shape[1]
    item_variances = df_items.var(axis=0, ddof=1).sum()
    total_variance = df_items.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - (item_variances / total_variance))

alpha = cronbach_alpha(items_data)
print(f"Overall AGS Scale Cronbach's Alpha (15 items): {alpha:.3f}")
print(f"Mean AGS Total Score: {ags_df['AGS_Total_Mean'].mean():.2f} (SD = {ags_df['AGS_Total_Mean'].std():.2f})")


## 3. AGS Distribution Plot

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(ags_df['AGS_Total_Mean'], kde=True, color='teal', bins=12)
plt.title('Distribution of Authenticity Gap Scale (AGS) Mean Scores', fontsize=12)
plt.xlabel('AGS Mean Score (1-5 Likert)')
plt.ylabel('Participant Count')
plt.axvline(ags_df['AGS_Total_Mean'].mean(), color='red', linestyle='--', label=f"Mean = {ags_df['AGS_Total_Mean'].mean():.2f}")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

ling_df = pd.read_csv('linguistic_features_120.csv')
print(ling_df.info())
print(ling_df.head())


## 1. Paired T-Tests (Condition: Raw vs AI)

In [ ]:
raw = ling_df[ling_df['Condition_Code'] == 0].set_index('Participant_ID')
ai = ling_df[ling_df['Condition_Code'] == 1].set_index('Participant_ID')

features = [
    'Grammar_Errors',
    'Error_Free_T_Units_Pct',
    'MTLD_Lexical_Diversity',
    'Lexical_Overlap_Pct',
    'Stance_Markers_Per_100w',
    'First_Person_Expressions_Per_100w'
]

results = []
for feat in features:
    r_vals = raw[feat]
    a_vals = ai[feat]
    t_val, p_val = stats.ttest_rel(r_vals, a_vals)
    diff = a_vals - r_vals
    d = diff.mean() / diff.std()
    results.append({
        'Feature': feat,
        'Raw_Mean': r_vals.mean(),
        'Raw_SD': r_vals.std(),
        'AI_Mean': a_vals.mean(),
        'AI_SD': a_vals.std(),
        't_stat': t_val,
        'p_value': p_val,
        'Cohens_d': d
    })

results_df = pd.DataFrame(results)
display_cols = ['Feature', 'Raw_Mean', 'Raw_SD', 'AI_Mean', 'AI_SD', 't_stat', 'p_value', 'Cohens_d']
print(results_df[display_cols].round(3).to_string(index=False))


## 2. Comparative Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

sns.boxplot(data=ling_df, x='Condition', y='Grammar_Errors', ax=axes[0], palette='Set2')
axes[0].set_title('Grammar Errors per Text')

sns.boxplot(data=ling_df, x='Condition', y='MTLD_Lexical_Diversity', ax=axes[1], palette='Set2')
axes[1].set_title('Lexical Diversity (MTLD)')

sns.boxplot(data=ling_df, x='Condition', y='Stance_Markers_Per_100w', ax=axes[2], palette='Set2')
axes[2].set_title('Authorial Stance Markers / 100w')

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

lmem_df = pd.read_csv('lmem_full_data.csv')
print(lmem_df.head())


## 1. Fit Linear Mixed-Effects Model (MTLD)

In [ ]:
model_mtld = smf.mixedlm(
    "MTLD_Lexical_Diversity ~ Condition_Code * CEFR_Numeric + AI_Experience_Months",
    data=lmem_df,
    groups=lmem_df["Participant_ID"],
    re_formula="~Condition_Code"
)
fit_mtld = model_mtld.fit()
print(fit_mtld.summary())


## 2. Fit Linear Mixed-Effects Model (Grammar Errors)

In [ ]:
model_err = smf.mixedlm(
    "Grammar_Errors ~ Condition_Code * CEFR_Numeric + AI_Experience_Months",
    data=lmem_df,
    groups=lmem_df["Participant_ID"],
    re_formula="~Condition_Code"
)
fit_err = model_err.fit()
print(fit_err.summary())


## 3. Load & Inspect Manuscript Table 5 (Published LMEM Summary)

In [ ]:
t5_summary = pd.read_csv('lmem_results_t5.csv')
print(t5_summary)
